In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_parquet('../data/processed/iran_films_principals.parquet')

In [3]:
df.head()

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres,ordering,nconst,category,job,characters
0,tt0044728,movie,Huracán Ramírez,Huracán Ramírez,0,1953,\N,103,"Action,Comedy,Crime",1,nm0798270,actor,\N,"[""Fernando Torres (Huracán Ramírez)""]"
1,tt0044728,movie,Huracán Ramírez,Huracán Ramírez,0,1953,\N,103,"Action,Comedy,Crime",2,nm0739131,actress,\N,"[""Margarita""]"
2,tt0044728,movie,Huracán Ramírez,Huracán Ramírez,0,1953,\N,103,"Action,Comedy,Crime",3,nm0327643,actress,\N,"[""Laura""]"
3,tt0044728,movie,Huracán Ramírez,Huracán Ramírez,0,1953,\N,103,"Action,Comedy,Crime",4,nm0273508,actor,\N,"[""El pichi""]"
4,tt0044728,movie,Huracán Ramírez,Huracán Ramírez,0,1953,\N,103,"Action,Comedy,Crime",5,nm0414109,actor,\N,"[""Señor Torres (Tonina Jackson)""]"


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 92518 entries, 0 to 92517
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   tconst          92518 non-null  object
 1   titleType       92518 non-null  object
 2   primaryTitle    92518 non-null  object
 3   originalTitle   92518 non-null  object
 4   isAdult         92518 non-null  int64 
 5   startYear       92518 non-null  object
 6   endYear         92518 non-null  object
 7   runtimeMinutes  92518 non-null  object
 8   genres          92518 non-null  object
 9   ordering        92518 non-null  int64 
 10  nconst          92518 non-null  object
 11  category        92518 non-null  object
 12  job             92518 non-null  object
 13  characters      92518 non-null  object
dtypes: int64(2), object(12)
memory usage: 9.9+ MB


In [5]:
df = df.replace('\\N', np.nan)

In [6]:
missing_report = pd.DataFrame({
    'column': df.columns,
    'missing_count': df.isna().sum().values,
    'missing_rate': df.isna().mean().values
}).sort_values('missing_rate', ascending=False)

print(missing_report)

            column  missing_count  missing_rate
12             job          83543      0.902992
6          endYear          80248      0.867377
13      characters          76715      0.829190
7   runtimeMinutes          43742      0.472794
8           genres          29148      0.315052
5        startYear            429      0.004637
4          isAdult              0      0.000000
3    originalTitle              0      0.000000
2     primaryTitle              0      0.000000
1        titleType              0      0.000000
0           tconst              0      0.000000
9         ordering              0      0.000000
11        category              0      0.000000
10          nconst              0      0.000000


 هدف تحلیل شبکه همکاری است به این مقادیر نیازی نداریم.

In [7]:
df = df.drop(columns=['job','characters'])

In [8]:
# modifying the data type of numeric columns
df['startYear'] = pd.to_numeric(df['startYear'], errors='coerce')
df['runtimeMinutes'] = pd.to_numeric(df['runtimeMinutes'], errors='coerce')

# drop records that missed the 'startYear' value(because they're less than 0.5%)
df = df.dropna(subset=['startYear'])
df['startYear'] = df['startYear'].astype(int)

سال هایی که خارج از بازه منطقی هستند را حذف میکنیم

In [9]:
print(df['startYear'].min(), df['startYear'].max())

1930 2026


In [10]:
df = df[(df['startYear'] >= 1900) & (df['startYear'] <= 2026)]

حذف رکورد های تکراری کامل

In [11]:
duplicate_count = df.duplicated().sum()
print(duplicate_count)

0


In [12]:
df = df.drop_duplicates()

ببینیم نقش ها چه مقادیری دارند

In [13]:
df['category'].value_counts()

category
actor                  42017
actress                19599
director                6982
writer                  5421
producer                4826
cinematographer         4513
editor                  3803
composer                2515
self                    1403
production_designer      821
archive_footage          108
casting_director          67
archive_sound             14
Name: count, dtype: int64

فقط نقش های مرتبط با تحلیل شبکه همکاری را نگه میداریم 

In [14]:
relevant_roles = ['actor', 'actress', 'director', 'writer']
df = df[df['category'].isin(relevant_roles)]
print(df.shape)

(74019, 12)


پیدا کردن اسم افراد که در جدول دیگری قرار داره 
بعد هم چک میکنیم ببینیم که ایا کسی هست شماره اش باشه ولی اسمش توی جدول نباشه

In [15]:
df_names = pd.read_csv('../data/raw/name.basics.tsv.gz', sep='\t', usecols=['nconst', 'primaryName'])

In [16]:
df = df.merge(df_names, on='nconst', how='left')

In [17]:
print(df['primaryName'].isna().sum())

7


In [18]:
df.to_parquet('../data/processed/iran_cinema_clean.parquet')
print(df.shape)
df.head()

(74019, 13)


,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres,ordering,nconst,category,primaryName
0,tt0044728,movie,Huracán Ramírez,Huracán Ramírez,0,1953,NaN,103.0,"Action,Comedy,Crime",1,nm0798270,actor,David Silva
1,tt0044728,movie,Huracán Ramírez,Huracán Ramírez,0,1953,NaN,103.0,"Action,Comedy,Crime",2,nm0739131,actress,Titina Romay
2,tt0044728,movie,Huracán Ramírez,Huracán Ramírez,0,1953,NaN,103.0,"Action,Comedy,Crime",3,nm0327643,actress,Carmelita González
3,tt0044728,movie,Huracán Ramírez,Huracán Ramírez,0,1953,NaN,103.0,"Action,Comedy,Crime",4,nm0273508,actor,Freddy Fernández
4,tt0044728,movie,Huracán Ramírez,Huracán Ramírez,0,1953,NaN,103.0,"Action,Comedy,Crime",5,nm0414109,actor,Tonina Jackson
